In [ ]:
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, silhouette_score
import pandas as pd
from datetime import datetime
import numpy as np
import matplotlib.pylab as plt
from matplotlib.pylab import rcParams


Notes on process:

- I'm clustering only weather variables and all of them, i.e. for wind and sun conditions. 
    - Weather is the factor determining your grouping and power is the output of whatever that grouping ends up looking like. 
    - Each cluster is a different weather scenario => having all weather conditions together as input enables you to cover all the spectrum of possibilities.
- The variables that you include in the weather-defining factors are important and not all need be there. I'll progressively decide on whoch ones to incorporate

In [ ]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

final_file = "data/complete_dataframe_30min.csv"
df = pd.read_csv(str(ROOT/final_file), delimiter=',', decimal='.', parse_dates=['ts'], index_col='ts') #ts is already the index :)

In [ ]:
# Column names for reference
df.columns

In [ ]:
#Lines that I need to add –– I just did, so don't run this

#does df have missing values (NaNs)?
print(df_model.shape)
print(df[required_cols].isna().sum())

#what is the size of the clusters? The smaller they are, the less reliable: "when choosing K, do not look only at lowest MAE. 
# Also check whether the smallest cluster becomes tiny. If a candidate K gives slightly lower MAE but has clusters of size 2 or 3, 
# I would be cautious."

print("\n=== Cluster sizes in best training model ===")
print(best_model['train_labeled']['cluster'].value_counts().sort_index())

In [ ]:
# ============================================================
# 1. USER SETTINGS
# ============================================================

# Name of timestamp column
timestamp_col = 'ts'   # change this to your actual time column name

# Whether to resample
do_resample = False

# Examples:
# '15min', '30min', '1H', '3H', '1D'
resample_rule = '1H'

# How to aggregate during resampling
# Usually 'mean' is the right starting point for weather and power time series
resample_agg = 'mean'

# Weather features to use for clustering
weather_cols = [
    'radia_glob',
    'wind_speed',
    'temp_dry',
]

# Optional: include wind direction properly encoded
use_wind_direction = False
wind_dir_col = 'wind_dir'

# PV columns to aggregate
pv_cols = [
    'B117_m',
    'B319_m',
    'B330_1_m',
    'B330_2_m',
    'B330_3_m',
    'B716_m',
    'B715_m',
]

# Wind power columns to aggregate
wind_power_cols = [
    'Aircon_WT Power_m',
    'Gaia_WT Power_m',
]

# Multi-fold settings
n_time_blocks = 6         # split full time series into this many contiguous blocks
min_train_blocks = 2      # first folds need at least this many train blocks

# Range of K values to test
k_values = range(2, 12)

# Random seed
random_state = 42

In [ ]:
# 2. HELPER FUNCTIONS

def add_wind_direction_encoding(df, wind_dir_col):
    """
    Encode wind direction as sin/cos to handle circularity.
    """
    df = df.copy()
    radians = np.deg2rad(df[wind_dir_col])
    df['wind_dir_sin'] = np.sin(radians)
    df['wind_dir_cos'] = np.cos(radians)
    return df


def resample_dataframe(df, timestamp_col, rule='1H', agg='mean'):
    """
    Resample dataframe by time.
    Timestamp is datetime index of df.
    Only numeric columns are retained by the aggregation.
    """
    df_resampled = df.copy()
    df_resampled.index = pd.to_datetime(df_resampled.index)

    if agg == 'mean':
        df_resampled = df_resampled.resample(rule).mean()
    elif agg == 'sum':
        df_resampled = df_resampled.resample(rule).sum()
    elif agg == 'median':
        df_resampled = df_resampled.resample(rule).median()
    else:
        raise ValueError("agg must be one of: 'mean', 'sum', 'median'")

    return df_resampled

def prepare_dataframe(df):
    """
    Prepare a modeling dataframe:
    - optionally encode wind direction
    - create aggregated power columns
    - remove rows with missing required data
    """
    df_model = df.copy()

    # Aggregate plant outputs
    df_model['PV_total'] = df_model[pv_cols].sum(axis=1, min_count=1)
    df_model['WT_total'] = df_model[wind_power_cols].sum(axis=1, min_count=1)
    df_model['Power_total'] = df_model[['PV_total', 'WT_total']].sum(axis=1, min_count=1)

    feature_cols = weather_cols.copy()

    if use_wind_direction:
        df_model = add_wind_direction_encoding(df_model, wind_dir_col)
        feature_cols.extend(['wind_dir_sin', 'wind_dir_cos'])

    required_cols = feature_cols + ['PV_total', 'WT_total', 'Power_total']
    df_model = df_model.dropna(subset=required_cols).copy()

    return df_model, feature_cols

def make_time_block_folds(df, n_time_blocks=6, min_train_blocks=2):
    """
    Create expanding-window time folds from contiguous blocks.

    Example with n_time_blocks=6:
      block 0 | block 1 | block 2 | block 3 | block 4 | block 5

    Folds:
      train=[0,1],       test=[2]
      train=[0,1,2],     test=[3]
      train=[0,1,2,3],   test=[4]
      train=[0,1,2,3,4], test=[5]

    Returns:
      list of (train_idx, test_idx)
    """
    n = len(df)

    block_sizes = np.full(n_time_blocks, n // n_time_blocks)
    block_sizes[: n % n_time_blocks] += 1

    starts = np.cumsum(np.r_[0, block_sizes[:-1]])
    ends = np.cumsum(block_sizes)

    blocks = [np.arange(start, end) for start, end in zip(starts, ends)]

    folds = []
    for test_block in range(min_train_blocks, n_time_blocks):
        train_idx = np.concatenate(blocks[:test_block])
        test_idx = blocks[test_block]
        folds.append((train_idx, test_idx))

    return folds


def evaluate_kmeans_across_folds(df_model, feature_cols, k, folds, random_state=42):
    """
    Evaluate one K across multiple time folds and average the results.
    """
    fold_metrics = []
    fold_models = []

    for fold_num, (train_idx, test_idx) in enumerate(folds, start=1):
        train_df = df_model.iloc[train_idx].copy()
        test_df = df_model.iloc[test_idx].copy()

        metrics, train_labeled, test_labeled, cluster_power, scaler, kmeans = evaluate_kmeans_for_k(
            train_df=train_df,
            test_df=test_df,
            feature_cols=feature_cols,
            k=k,
            random_state=random_state
        )

        metrics['fold'] = fold_num
        metrics['train_rows'] = len(train_df)
        metrics['test_rows'] = len(test_df)

        fold_metrics.append(metrics)
        fold_models.append({
            'fold': fold_num,
            'train_labeled': train_labeled,
            'test_labeled': test_labeled,
            'cluster_power': cluster_power,
            'scaler': scaler,
            'kmeans': kmeans,
        })

    fold_metrics_df = pd.DataFrame(fold_metrics)

    summary = {
        'K': k,
        'mean_train_inertia': fold_metrics_df['train_inertia'].mean(),
        'mean_train_silhouette': fold_metrics_df['train_silhouette'].mean(),
        'mean_test_mae_pv': fold_metrics_df['test_mae_pv'].mean(),
        'mean_test_rmse_pv': fold_metrics_df['test_rmse_pv'].mean(),
        'mean_test_mae_wt': fold_metrics_df['test_mae_wt'].mean(),
        'mean_test_rmse_wt': fold_metrics_df['test_rmse_wt'].mean(),
        'mean_test_mae_total': fold_metrics_df['test_mae_total'].mean(),
        'mean_test_rmse_total': fold_metrics_df['test_rmse_total'].mean(),
        'mean_min_train_cluster_size': fold_metrics_df['min_train_cluster_size'].mean(),
        'mean_max_train_cluster_size': fold_metrics_df['max_train_cluster_size'].mean(),
        'std_test_mae_total': fold_metrics_df['test_mae_total'].std(),
        'n_folds': len(fold_metrics_df),
    }

    return summary, fold_metrics_df, fold_models


def fit_final_kmeans_on_all_data(df_model, feature_cols, k, random_state=42):
    """
    Fit final descriptive model on all available data after best K is selected.
    This is for cluster descriptions / scenario interpretation, not unbiased testing.
    """
    scaler = StandardScaler()
    X_all = scaler.fit_transform(df_model[feature_cols])

    kmeans = KMeans(
        n_clusters=k,
        random_state=random_state,
        n_init=10
    )

    all_clusters = kmeans.fit_predict(X_all)

    all_labeled = df_model.copy()
    all_labeled['cluster'] = all_clusters

    cluster_power = all_labeled.groupby('cluster')[['PV_total', 'WT_total', 'Power_total']].mean()

    return all_labeled, cluster_power, scaler, kmeans

def safe_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def evaluate_kmeans_for_k(train_df, test_df, feature_cols, k, random_state=42):
    """
    Train/evaluate one KMeans model for one value of K.
    """
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_df[feature_cols])
    X_test = scaler.transform(test_df[feature_cols])

    kmeans = KMeans(
        n_clusters=k,
        random_state=random_state,
        n_init=10
    )

    train_clusters = kmeans.fit_predict(X_train)
    test_clusters = kmeans.predict(X_test)

    train_labeled = train_df.copy()
    test_labeled = test_df.copy()

    train_labeled['cluster'] = train_clusters
    test_labeled['cluster'] = test_clusters

    # Learn average power from TRAIN only
    cluster_power = train_labeled.groupby('cluster')[['PV_total', 'WT_total', 'Power_total']].mean()

    # Predict TEST by cluster lookup
    for target in ['PV_total', 'WT_total', 'Power_total']:
        test_labeled[f'{target}_pred'] = test_labeled['cluster'].map(cluster_power[target])

    metrics = {
        'K': k,
        'train_inertia': kmeans.inertia_,
        'train_silhouette': np.nan,
        'test_mae_pv': mean_absolute_error(test_labeled['PV_total'], test_labeled['PV_total_pred']),
        'test_rmse_pv': safe_rmse(test_labeled['PV_total'], test_labeled['PV_total_pred']),
        'test_mae_wt': mean_absolute_error(test_labeled['WT_total'], test_labeled['WT_total_pred']),
        'test_rmse_wt': safe_rmse(test_labeled['WT_total'], test_labeled['WT_total_pred']),
        'test_mae_total': mean_absolute_error(test_labeled['Power_total'], test_labeled['Power_total_pred']),
        'test_rmse_total': safe_rmse(test_labeled['Power_total'], test_labeled['Power_total_pred']),
        'min_train_cluster_size': pd.Series(train_clusters).value_counts().min(),
        'max_train_cluster_size': pd.Series(train_clusters).value_counts().max(),
    }

    unique_clusters = np.unique(train_clusters)
    if len(unique_clusters) > 1 and len(train_labeled) > len(unique_clusters):
        metrics['train_silhouette'] = silhouette_score(X_train, train_clusters)

    return metrics, train_labeled, test_labeled, cluster_power, scaler, kmeans


def plot_k_comparison(results_df):
    """
    Improved plot: clearer distinction between mean test MAE and mean silhouette.
    """
    fig, ax1 = plt.subplots(figsize=(9, 5))

    line1 = ax1.plot(
        results_df['K'],
        results_df['mean_test_mae_total'],
        marker='o',
        linestyle='-',
        linewidth=2,
        label='Mean Test MAE (Total Power)'
    )

    ax1.set_xlabel('Number of clusters (K)')
    ax1.set_ylabel('Mean Test MAE (Total Power)')
    ax1.tick_params(axis='y')

    ax2 = ax1.twinx()

    line2 = ax2.plot(
        results_df['K'],
        results_df['mean_train_silhouette'],
        marker='s',
        linestyle='--',
        linewidth=2,
        label='Mean Train Silhouette Score'
    )

    ax2.set_ylabel('Mean Silhouette Score')
    ax2.tick_params(axis='y')

    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc='best')

    best_k = results_df.loc[results_df['mean_test_mae_total'].idxmin(), 'K']
    ax1.axvline(best_k, linestyle=':', linewidth=1)
    ax1.text(best_k, ax1.get_ylim()[1], f'  best K={best_k}', verticalalignment='top')

    ax1.set_title('K comparison: Mean prediction error vs clustering quality across folds')

    plt.tight_layout()
    plt.show()

def predict_power_for_new_weather(new_weather_df, feature_cols, scaler, kmeans, cluster_power):
    """
    Predict power for new weather scenarios.
    """
    X_new = scaler.transform(new_weather_df[feature_cols])
    new_clusters = kmeans.predict(X_new)

    pred_df = new_weather_df.copy()
    pred_df['cluster'] = new_clusters
    pred_df['PV_total_pred'] = pd.Series(new_clusters).map(cluster_power['PV_total']).values
    pred_df['WT_total_pred'] = pd.Series(new_clusters).map(cluster_power['WT_total']).values
    pred_df['Power_total_pred'] = pd.Series(new_clusters).map(cluster_power['Power_total']).values

    return pred_df

def compute_cluster_error_stats(test_labeled):
    """
    Compute per-cluster error statistics on the TEST set.
    Assumes test_labeled contains:
      - cluster
      - actual columns: PV_total, WT_total, Power_total
      - prediction columns: PV_total_pred, WT_total_pred, Power_total_pred
    """
    df_err = test_labeled.copy()

    # Absolute errors
    df_err['PV_abs_error'] = (df_err['PV_total'] - df_err['PV_total_pred']).abs()
    df_err['WT_abs_error'] = (df_err['WT_total'] - df_err['WT_total_pred']).abs()
    df_err['Power_abs_error'] = (df_err['Power_total'] - df_err['Power_total_pred']).abs()

    cluster_stats = df_err.groupby('cluster').agg(
        test_count=('cluster', 'size'),
        pv_mae=('PV_abs_error', 'mean'),
        wt_mae=('WT_abs_error', 'mean'),
        power_mae=('Power_abs_error', 'mean'),
        pv_actual_mean=('PV_total', 'mean'),
        pv_pred_mean=('PV_total_pred', 'mean'),
        wt_actual_mean=('WT_total', 'mean'),
        wt_pred_mean=('WT_total_pred', 'mean'),
        power_actual_mean=('Power_total', 'mean'),
        power_pred_mean=('Power_total_pred', 'mean'),
    ).reset_index()

    return cluster_stats

def plot_cluster_mae_with_counts(cluster_stats, sort_by='power_mae'):
    """
    Plot per-cluster MAE with test sample counts overlaid.
    """
    plot_df = cluster_stats.sort_values(sort_by).copy()

    fig, ax1 = plt.subplots(figsize=(10, 5))

    # --- Bar plot (MAE) ---
    bars = ax1.bar(
        plot_df['cluster'].astype(str),
        plot_df['power_mae'],
        label='MAE (Total Power)'
    )

    ax1.set_xlabel('Cluster')
    ax1.set_ylabel('MAE (Total Power)')
    ax1.set_title('Per-cluster MAE and test sample count for best K')

    # --- Line plot (counts) ---
    ax2 = ax1.twinx()
    line = ax2.plot(
        plot_df['cluster'].astype(str),
        plot_df['test_count'],
        marker='o',
        linestyle='--',
        linewidth=2,
        color='pink',
        label='Test sample count'
    )

    ax2.set_ylabel('Test sample count')

    # --- Combine legends ---
    handles1, labels1 = ax1.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()

    ax1.legend(handles1 + handles2, labels1 + labels2, loc='best')

    plt.tight_layout()
    plt.show()

def build_cluster_descriptions(train_labeled, feature_cols):
    """
    Build a per-cluster summary from TRAIN data so every learned cluster
    gets a description, even if it does not appear in the TEST set.
    """
    cluster_desc = train_labeled.groupby('cluster').agg(
        cluster_size=('cluster', 'size'),
        **{col: (col, 'mean') for col in feature_cols}
    ).reset_index()

    return cluster_desc



In [ ]:
# ============================================================
# 3. MAIN WORKFLOW –– Preparing dataframe
# ============================================================

# Make a working copy
df_work = df.copy()

# Double check index is datetime and sort by it
df_work.index = pd.to_datetime(df_work.index)
df_work = df_work.sort_index()

# Optional resampling
if do_resample:
    df_work = resample_dataframe(
        df_work,
        timestamp_col=timestamp_col,
        rule=resample_rule,
        agg=resample_agg
    )
    print(f"Data resampled using rule={resample_rule}, agg={resample_agg}")

    # If resampling reset the index into a column, restore datetime index cleanly
    if timestamp_col in df_work.columns:
        df_work[timestamp_col] = pd.to_datetime(df_work[timestamp_col])
        df_work = df_work.set_index(timestamp_col).sort_index()

# Prepare data
df_model, feature_cols = prepare_dataframe(df_work)

# Ensure chronological order
df_model = df_model.sort_index()

print(f"Total rows after cleaning: {len(df_model)}")
print(f"Weather features used: {feature_cols}")

# ============================================================
# 3. MAIN WORKFLOW –– Multi-fold evaluation across K
# ============================================================

folds = make_time_block_folds(
    df_model,
    n_time_blocks=n_time_blocks,
    min_train_blocks=min_train_blocks
)

print(f"Number of evaluation folds: {len(folds)}")
for i, (train_idx, test_idx) in enumerate(folds, start=1):
    print(f"Fold {i}: train_rows={len(train_idx)}, test_rows={len(test_idx)}")

results = []
saved_models = {}

for k in k_values:
    summary, fold_metrics_df, fold_models = evaluate_kmeans_across_folds(
        df_model=df_model,
        feature_cols=feature_cols,
        k=k,
        folds=folds,
        random_state=random_state
    )

    results.append(summary)
    saved_models[k] = {
        'fold_metrics_df': fold_metrics_df,
        'fold_models': fold_models,
    }

results_df = pd.DataFrame(results).sort_values('K').reset_index(drop=True)

# Choose best K using mean test MAE across folds
best_k = results_df.loc[results_df['mean_test_mae_total'].idxmin(), 'K']
best_k_models = saved_models[best_k]

print("\n=== Comparison of K values across folds ===")
print(results_df)

# Plot tradeoff
plot_k_comparison(results_df)

print(f"\nBest K based on lowest mean_test_mae_total across folds: {best_k}")

print("\n=== Best-K summary ===")
print(results_df[results_df['K'] == best_k].reset_index(drop=True))

print("\n=== Fold-by-fold metrics for best K ===")
print(best_k_models['fold_metrics_df'])

# ============================================================
# 3A. FIT FINAL DESCRIPTIVE MODEL ON ALL DATA USING BEST K
# ============================================================

final_labeled, final_cluster_power, final_scaler, final_kmeans = fit_final_kmeans_on_all_data(
    df_model=df_model,
    feature_cols=feature_cols,
    k=best_k,
    random_state=random_state
)

print("\n=== Cluster sizes in final all-data model ===")
print(final_labeled['cluster'].value_counts().sort_index())

print("\n=== Cluster-average power table for final all-data model ===")
print(final_cluster_power)

best_cluster_descriptions = build_cluster_descriptions(
    final_labeled,
    feature_cols
)

print("\n=== Cluster descriptions for all clusters in final all-data model ===")
print(best_cluster_descriptions.sort_values('cluster'))

# ============================================================
# 3B. PER-CLUSTER ERROR ANALYSIS FOR BEST K
# Use the best-performing fold for detailed per-cluster TEST analysis
# ============================================================

best_fold_idx = best_k_models['fold_metrics_df']['test_mae_total'].idxmin()
best_fold_number = best_k_models['fold_metrics_df'].loc[best_fold_idx, 'fold']
best_fold_model = best_k_models['fold_models'][best_fold_number - 1]

best_cluster_stats = compute_cluster_error_stats(best_fold_model['test_labeled'])

# Add cluster descriptions from FINAL all-data model
best_cluster_stats = best_cluster_stats.merge(
    best_cluster_descriptions,
    on='cluster',
    how='left'
)

print(f"\n=== Using fold {best_fold_number} for detailed per-cluster TEST error analysis ===")
print("\n=== Per-cluster TEST error stats for best K ===")
print(best_cluster_stats.sort_values('power_mae'))

# Plot per-cluster MAE and size
plot_cluster_mae_with_counts(best_cluster_stats, sort_by='power_mae')

What are we evaluating with the above metrics? 

- Does the number of K's we have lead to meaningful clusters?
- How well are the clusters predicting power?

NOtes:

RMSE and MAE are prediction metrics:
    - MAE is measure is same unit of power and tells you by how much your predictions are off on average
    - If RMSE >> MAE → you have outliers or bad clusters

Inertia is a clustering metric:
- Inertia: Sum of squared distances to cluster centers
    *lower means tighter clusters
- Train silhouette measures how well-separated the clusters are
    +1 → well separated
    0 → overlapping
    -1 → badly assigned

Cluster size shouldn't become unreasonably small––shrinking makes the average across it unreliable

When we go up to K = 10, K = 15, or K = 20, the best K remains K = 10:
- Stats: 
    Best K based on lowest test_mae_total: 10

    === Cluster-average power table for best K ===
            PV_total  WT_total  Power_total
    cluster                                  
    0         1.675372  0.341383     2.016755
    1         1.764595  2.848348     4.612943
    2        19.924367  0.680930    20.605298
    3        37.341426  1.168693    38.510119
    4         1.200963  0.456910     1.657873
    5         3.078054  7.833272    10.911326
    6         3.443237  2.617929     6.061166
    7         1.209053  0.187315     1.396368
    8        19.980325  4.870680    24.851005
    9        39.572776  6.171012    45.743788

    === Best-K summary ===
        K  train_inertia  train_silhouette  test_mae_pv  test_rmse_pv  \
    8  10    5734.481891          0.284789     2.406483      3.577235   

    test_mae_wt  test_rmse_wt  test_mae_total  test_rmse_total  \
    8     1.505428      2.015055        2.567269         4.038542   

    min_train_cluster_size  max_train_cluster_size  
    8                     487                    1850 

    In per-cluster analysis once scenario-based approximations are made on the TEST data show only 4 of the clusters becuase those are the clusters that the smaller test data set maps out to!

If we stay below K = 9, the best is K = 5





In [ ]:
# ============================================================
# 4. OPTIONAL: FUTURE SCENARIO PREDICTION
# ============================================================

# Example:
# future_weather = pd.DataFrame({
#     'radia_glob': [500, 900, 50],
#     'wind_speed': [4.2, 7.1, 11.5],
#     'temp_dry': [12.0, 21.0, 6.0],
# })
#
# if use_wind_direction:
#     future_weather['wind_dir'] = [45, 180, 300]
#     future_weather = add_wind_direction_encoding(future_weather, wind_dir_col)
#
# future_predictions = predict_power_for_new_weather(
#     new_weather_df=future_weather,
#     feature_cols=feature_cols,
#     scaler=best_model['scaler'],
#     kmeans=best_model['kmeans'],
#     cluster_power=best_model['cluster_power']
# )
#
# print("\n=== Future scenario predictions ===")
# print(future_predictions)